# EDA — Tanzania perinatal cohort

We look at all pregnancies of 7+ months in the last five years and ask which
pregnancy-time factors separate the ones that end in a **perinatal death**
(stillbirth or death in the first 7 days) from the ones that don't.


## 0 · Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

DATA = '../../../data/processed/tz_perinatal.csv'
df = pd.read_csv(DATA)
df.shape


## 1 · First look


In [ ]:
df.head()


In [ ]:
df.info()


## 2 · Which columns do we actually use?

The file has 40 columns, but most are IDs, survey-design codes, raw DHS source
columns (already turned into the friendly ones), or outcome columns. Only **12**
are predictors we feed a model. We also keep `v001` and `v005` — not as features,
but because the modelling stage needs them for grouped cross-validation and for
the survey weight.


In [ ]:
target = 'perinatal_death'

predictors = [
    'maternal_age', 'prior_pregnancies', 'prior_live_births',
    'prior_stillbirth', 'prior_loss', 'preceding_interval_m',
    'multiple', 'education', 'wealth_quintile', 'residence',
    'region', 'told_high_bp',
]
design = ['v001', 'v005', 'weight']            # kept for CV + weighting, not features
outcome_cols = ['stillbirth', 'early_neonatal_death', 'perinatal_death',
                'pregnancy_outcome', 'pregnancy_months', 'baby_sex']

dropped = [c for c in df.columns if c not in predictors + design + outcome_cols]
print(f'total columns   : {df.shape[1]}')
print(f'predictors used : {len(predictors)}')
print(f'outcome columns : {len(outcome_cols)}')
print(f'design (keep)   : {len(design)}')
print(f'dropped (IDs/raw): {len(dropped)}')
print('\ndropped:', dropped)


In [ ]:
numeric_preds = ['maternal_age', 'prior_pregnancies', 'prior_live_births',
                 'preceding_interval_m']
cat_preds = ['prior_stillbirth', 'prior_loss', 'multiple', 'education',
             'wealth_quintile', 'residence', 'told_high_bp']

df[predictors + [target]].describe(include='all').T


## 3 · Missing values


In [ ]:
missing = (df[predictors].isna().mean() * 100).round(1)
missing[missing > 0]


Only `preceding_interval_m` is missing (~21%), and that is **first pregnancies** —
there is no preceding birth to measure an interval from. We don't drop these moms.
Instead we add a `first_pregnancy` flag (so the model keeps that information) and
fill the interval with the median of the remaining pregnancies.


In [ ]:
df['first_pregnancy'] = df['preceding_interval_m'].isna().astype(int)
df['preceding_interval_m'] = df['preceding_interval_m'].fillna(
    df['preceding_interval_m'].median())

print('first pregnancies:', int(df['first_pregnancy'].sum()))
print('remaining missing in predictors:', int(df[predictors].isna().sum().sum()))
print('duplicate rows:', df.duplicated().sum())


## 4 · The outcome


In [ ]:
n = len(df); events = int(df[target].sum())
print(f'perinatal deaths: {events} / {n} = {events/n:.2%}')
print('  stillbirths:', int(df['stillbirth'].sum()),
      '| early neonatal:', int(df['early_neonatal_death'].sum()))

w = df['v005'] / 1_000_000
print(f'weighted rate: {(df[target]*w).sum()/w.sum():.2%}')


## 5 · Univariate — each variable on its own


In [ ]:
df[numeric_preds].hist(figsize=(11, 7), bins=30)
plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, c in zip(axes.ravel(), cat_preds):
    df[c].value_counts(dropna=False).plot(kind='bar', ax=ax, title=c)
for ax in axes.ravel()[len(cat_preds):]:
    ax.axis('off')
plt.tight_layout(); plt.show()


## 6 · Perinatal death rate by each predictor

The core question: where does the death rate climb above the ~3.75% baseline?


In [ ]:
for c in cat_preds:
    print(f'\n--- {c} (death rate %) ---')
    print((df.groupby(c)[target].mean() * 100).round(2))


In [ ]:
age_grp = pd.cut(df['maternal_age'], [0, 18, 35, 100], right=False, labels=['<18', '18-34', '35+'])
par_grp = pd.cut(df['prior_live_births'], [-1, 0, 4, 100], labels=['0', '1-4', '5+'])
print('age group:', (df.groupby(age_grp, observed=True)[target].mean()*100).round(2).to_dict())
print('parity   :', (df.groupby(par_grp, observed=True)[target].mean()*100).round(2).to_dict())
print('first pregnancy:', (df.groupby('first_pregnancy')[target].mean()*100).round(2).to_dict())


In [ ]:
strong = ['multiple', 'prior_stillbirth', 'told_high_bp', 'prior_loss']
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, c in zip(axes, strong):
    (df.groupby(c)[target].mean()*100).plot(kind='bar', ax=ax, title=c)
    ax.axhline(df[target].mean()*100, color='red', ls='--', lw=1)
axes[0].set_ylabel('perinatal death rate (%)')
plt.tight_layout(); plt.show()


## 7 · Correlations (numeric)


In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(df[numeric_preds + [target]].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', center=0)
plt.title('Numeric correlations'); plt.show()


## 8 · Findings

Baseline perinatal death rate is **3.75%** (412 of 10,980; 199 stillbirths + 213
early neonatal). The weighted rate is similar, so the sample is representative.

Factors that clearly raise risk:
- **Multiple pregnancy** — 13.7% vs 3.4% (strongest).
- **Previous stillbirth** — 9.8% vs 3.5%.
- **Told has high blood pressure** — 8.6% vs 3.5% (the hypertension angle; a self-report proxy).
- **Previous pregnancy loss** — 5.8% vs 3.4%.
- **Short interval (<24m)** and **first pregnancy** — both above baseline.
- **Age** — higher for <18 and 35+ than 18–34.

Caveat: risk looks higher for richer, urban, more-educated mothers. This is most
likely reporting bias — facility births are recorded better, so deaths are captured
more completely — not wealth causing risk. Flag before using wealth/residence.

Data handling: `preceding_interval_m` missing = first pregnancies → kept via a
`first_pregnancy` flag and median-filled. No duplicates.

For modelling: the history and clinical variables carry the signal and are all
genuinely pregnancy-time, so they are safe predictors.
